In [26]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from pathlib import Path
import datetime as dt
import time
import pytz
import json
import os

In [27]:
pst = pytz.timezone('America/Los_Angeles')
dt_str = dt.datetime.now(pst).strftime("%Y-%m-%d-%I_%M_%S_%p")
dt_str

'2026-08-21-01_03_05_AM'

In [28]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/data"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    output_path = "../../"

In [29]:
ss = pd.read_csv(f"{data_path}/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [ ]:
experiment_config = {
     "experiment": {
        "model": "ensemble",
        "experiment_oofs": ["2026-08-20-08_40_50_PM_xgboost", "2026-08-20-08_26_26_PM_xgboost", "2026-08-20-08_42_20_PM_xgboost", "2026-08-20-08_47_09_PM_xgboost", "2026-08-20-08_31_41_PM_xgboost"],
        "description": "5 raw xgboost auc mean ensemble with opt params",
    }
}
experiment_config["experiment"]["id"] = f"{dt_str}_{experiment_config["experiment"]["model"]}"

In [31]:
raw_train_df = pd.read_csv(f"{data_path}/raw/train.csv")
raw_train_id = raw_train_df["id"]
y = raw_train_df[target_column]

In [32]:
oof_dfs, ss_dfs = [], []

for fg in experiment_config["experiment"]["experiment_oofs"]:
    file_path = Path(output_path) / "experiments" / fg / "oof.csv"
    df = pd.read_csv(file_path)
    df.pop('id')
    oof_dfs.append(df)

for fg in experiment_config["experiment"]["experiment_oofs"]:
    file_path = Path(output_path) / "experiments" / fg / "submission.csv"
    df = pd.read_csv(file_path)
    df.pop('id')
    ss_dfs.append(df)

oof_df = pd.concat(oof_dfs, axis=1)
ss_df = pd.concat(ss_dfs, axis=1)
oof_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   addicted_label  691369 non-null  float64
 1   addicted_label  691369 non-null  float64
 2   addicted_label  691369 non-null  float64
 3   addicted_label  691369 non-null  float64
 4   addicted_label  691369 non-null  float64
dtypes: float64(5)
memory usage: 26.4 MB


In [33]:
y_oof_mean_cv = oof_df.mean(axis=1)
y_oof_mean_cv.rename(target_column, inplace=True)

y_ss_mean_cv = ss_df.mean(axis=1)

y_pred_df = pd.concat([raw_train_id, y_oof_mean_cv], axis=1)
ss[target_column] = y_ss_mean_cv

In [34]:
valid_auc_score = roc_auc_score(y, y_oof_mean_cv)
print("Validation Mean Ensemble AUC:", valid_auc_score)

Validation Mean Ensemble AUC: 0.967764171774383


In [35]:
metrics = {
    "experiment": dt_str + f"_{experiment_config["experiment"]["model"]}",
    "model": f"{experiment_config["experiment"]["model"]}",
    "experiment_oofs": f"{experiment_config["experiment"]["experiment_oofs"]}",
    "primary_metric": {
        "name": "auc",
        "value": round(valid_auc_score, 5)
    }
}

In [36]:
experiment_path = Path(output_path) / "experiments" / f"{dt_str}_{experiment_config["experiment"]["model"]}"
experiment_path.mkdir(parents=True, exist_ok=True)

with open(experiment_path / "full_config.json", "w") as f:
    json.dump(experiment_config, f, indent=4)

with open(experiment_path / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

y_pred_df.to_csv(experiment_path / "oof.csv", index=False)
ss.to_csv(experiment_path / f"submission.csv", index=False)